# T038: treino do adaptador LoRA no Google Colab

Treina o Qwen3.5-4B com QLoRA de 4 bits sobre o `tech4humans/br-doc-extraction` ([ADR-017](https://github.com/glauberg/visiondoc/blob/main/docs/decisions/ADR-017-treino-br-doc-extraction.md)) e gera, para a linha de base *zero-shot* e para o modelo treinado, o relatório com acurácia, F1 macro, matrizes de confusão e caracterização do ambiente (RF-SDD-18).

**Antes de começar**

- Ambiente de execução com GPU que suporte bfloat16 (L4 ou A100). A T4 não suporta bfloat16.
- O cache, os adaptadores e os relatórios ficam no Google Drive, para sobreviver ao reinício do ambiente.
- Etapas: (1) instalação; (2) preparação dos exemplos com PaddleOCR-VL; (3) **reiniciar o ambiente** para liberar a GPU usada pelo Paddle; (4) linha de base, treino e relatórios.

In [ ]:
!nvidia-smi

## 1. Instalação

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/visiondoc"
os.makedirs(BASE, exist_ok=True)

# Repositório privado: guarde um token do GitHub nos Secrets do Colab com o nome GITHUB_TOKEN.
try:
    from google.colab import userdata
    TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    TOKEN = None
URL = f"https://{TOKEN}@github.com/glauberg/visiondoc.git" if TOKEN else "https://github.com/glauberg/visiondoc.git"
if not os.path.exists("/content/visiondoc"):
    !git clone -q {URL} /content/visiondoc
%cd /content/visiondoc
!git pull -q
!pip install -q -e ".[treino]"

## 2. Preparação dos exemplos

Cada imagem passa por ingestão, pré-processamento, transcrição (PaddleOCR-VL), detecção e mascaramento, igual à inferência. O resultado vai para `BASE/cache` e a preparação é retomável: se o ambiente cair, rode de novo a partir daqui.

Confirme no guia de instalação do PaddleOCR a versão do `paddlepaddle-gpu` compatível com o CUDA do Colab.

In [ ]:
!pip install -q paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install -q "paddleocr[doc-parser]"

In [ ]:
from collections import Counter
from datasets import Image as ImagemHF, load_dataset

conjunto = load_dataset("tech4humans/br-doc-extraction")
conjunto = conjunto.cast_column("image", ImagemHF(decode=False))
print(conjunto)
linhas = [(particao, linha) for particao in ("train", "valid", "test") for linha in conjunto[particao]]
print(Counter((p, l["type"]) for p, l in linhas))

In [ ]:
from visiondoc.configuracao import carregar_configuracao
from visiondoc.deteccao import Detectores
from visiondoc.transcricao import (AvaliadorPorTranscricao, ClassificadorOrientacaoPaddle,
                                   TranscritorPaddleOCRVL, TranscritorPPOCR)
from visiondoc.treino.preparacao import preparar

cfg = carregar_configuracao("config/pipeline.yaml")
itens = preparar(
    linhas, f"{BASE}/cache", cfg,
    transcritor=TranscritorPaddleOCRVL(),
    avaliador=AvaliadorPorTranscricao(TranscritorPPOCR()),
    classificador=ClassificadorOrientacaoPaddle(),
    detectores=Detectores(None, None),  # sem llm-guard e ClamAV na preparação; regras e regex continuam
    regras_yara="config/yara/pdf_conteudo_ativo.yar",
)
print(Counter((i.particao, i.erro) for i in itens))

## 3. Reinicie o ambiente

Menu **Ambiente de execução → Reiniciar sessão**. Isso libera a memória de GPU ocupada pelo Paddle. Depois, continue na próxima célula (não é preciso repetir as etapas 1 e 2).

## 4. Linha de base, treino e relatórios

Para uma verificação rápida antes do treino completo, use `limite_treino=20, limite_teste=10`. Depois, rode com `None` para usar todas as linhas.

In [ ]:
%cd /content/visiondoc
import os
BASE = "/content/drive/MyDrive/visiondoc"
os.environ["MLFLOW_TRACKING_URI"] = f"file://{BASE}/mlruns"

from visiondoc.treino.executar import executar

resultado = executar(
    "config/pipeline.yaml", "config/treino.yaml",
    cache=f"{BASE}/cache", saida=f"{BASE}/saida", relatorios=f"{BASE}/relatorios",
    limite_treino=20, limite_teste=10,
)
resultado

## 5. Relatórios

In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display

for nome in ("zero_shot", "lora"):
    pasta = resultado.get(nome)
    if pasta is None:
        continue
    texto = (Path(pasta) / "relatorio.md").read_text(encoding="utf-8")
    display(Markdown(texto.split("## Classificação")[0]))
    for grafico in sorted((Path(pasta) / "graficos").glob("matriz_tipo_agregado_*.png")):
        display(Image(filename=str(grafico)))
print("Relatórios completos em", f"{BASE}/relatorios")